# multilingual-ner — Demo

Demonstrates all three components of the toolkit:
1. **Benchmark evaluation** — evaluate a NER model against a labelled dataset using seqeval
2. **Extraction** — run NER on unlabelled project data at scale
3. **Validation** — launch the Dash app for qualitative review

Uses dummy data throughout — no model downloads required for steps 1 and 3.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from multilingual_ner import ReadNERData
from multilingual_ner.extraction import NamedEntityExtractions

## 1 — Benchmark evaluation

Read a labelled NER dataset and evaluate a model against it using seqeval (entity-level) and sklearn (token-level) metrics.

In [ ]:
import tempfile, os

# Create a small dummy NER file in CoNLL BIO format
dummy_ner = """Ahmed B-PER
Younes I-PER
visited O
Cairo B-LOC
last O
week O
.

The O
UN B-ORG
met O
in O
Geneva B-LOC
.
"""

with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write(dummy_ner)
    tmp_path = f.name

reader = ReadNERData()
sentences, labels = reader.read_ner_file(tmp_path, sentence_boundary='.')
os.unlink(tmp_path)

print(f'Sentences: {len(sentences)}')
for sent, lbls in zip(sentences, labels):
    print(list(zip(sent, lbls)))

In [ ]:
from seqeval.metrics import classification_report

# Simulate model predictions (slightly imperfect)
true_labels  = [['B-PER', 'I-PER', 'O', 'B-LOC', 'O', 'O'],
                ['O', 'B-ORG', 'O', 'O', 'B-LOC']]
pred_labels  = [['B-PER', 'I-PER', 'O', 'B-LOC', 'O', 'O'],
                ['O', 'B-ORG', 'O', 'O', 'O']]  # missed Geneva

print(classification_report(true_labels, pred_labels))

## 2 — Extraction

Run NER on unlabelled project data using a pre-selected HuggingFace model.
Replace `model_name` with your chosen model (see language cards in `languages/`).

> **Note:** This cell downloads a model. Comment out if offline.

In [ ]:
# Uncomment to run with a real model
# df = pd.DataFrame({
#     'text': [
#         'Ahmed Younes visited Cairo last week.',
#         'The UN General Assembly met in New York.',
#         'Apple released a new product in Cupertino.',
#     ],
#     'message_id': ['msg_1', 'msg_2', 'msg_3'],
#     'accountId': ['acc_1', 'acc_1', 'acc_2'],
# })
#
# ner = NamedEntityExtractions(
#     model_name='dslim/bert-base-NER',  # English NER model
#     project_data=df,
#     text_col='text',
#     batch_size=8,
# )
#
# outputs_df = ner.extract_outputs()
# print(outputs_df[['text', 'PER', 'LOC', 'ORG', 'MISC']].head())

print('Uncomment the cells above to run extraction with a real model.')

## 3 — Validation app

Launch the Dash app to qualitatively review extraction outputs.
The app:
- Uploads a CSV/JSONL file with extraction results
- Displays sentences with colour-coded PER / LOC / ORG / MISC entities
- Lets reviewers annotate mistakes and missing entities
- Saves annotations to JSON

In [ ]:
# Run from terminal:
# python -m multilingual_ner.validation
# Then open http://localhost:8050

# Or launch inline (blocks the notebook):
# from multilingual_ner.validation import app
# app.run_server(debug=False)

print('Launch with: python -m multilingual_ner.validation')